# Dipole–dipole coupling between two spins

We calculate the dipolar interaction between two spin-1/2 particles:

$$
H_{\mathrm{dd}}
=
D_0\left[
\mathbf{S}_0\cdot\mathbf{S}_1
-
3(\mathbf{S}_0\cdot\hat{\mathbf{r}})
(\mathbf{S}_1\cdot\hat{\mathbf{r}})
\right].
$$

The coupling strength depends on the distance as $D_0\propto r^{-3}$.
Exchange interactions and tip fields are set to zero.

In [ ]:
# Imports
import numpy as np

from spinfinity import SpinSys, hamil

%load_ext autoreload
%autoreload 2

## Initialize two spins

The spins are separated along x. `AtomPositions` contains dimensionless
lattice coordinates, while `latticeLengthXY` contains spacings in nm.
The lattice spacing below is chosen for illustration.

In [ ]:
spin = SpinSys([0.5, 0.5])

spin.gfactorvector = [2.0, 2.0]
spin.B = [0.0, 0.0, 0.0]  # T
spin.BTip = [0.0, 0.0, 0.0]  # T
spin.Jvector[:] = 0.0  # No exchange interaction

# Physical positions = AtomPositions * latticeLengthXY
spin.latticeLengthXY = np.array([0.25, 0.25, 0.25])  # nm
spin.AtomPositions = np.array([
    [0.0, 0.0, 0.0],
    [4.0, 0.0, 0.0],
])

# Helper function to assign the dipolar coupling
# Calculate and assign the coupling from geometry and g-factors.
hamil.assign_DipolarCoupling(spin, [(0, 1, "calc")])

# Alternatively, assign a manually chosen coupling in meV:
# hamil.assign_DipolarCoupling(spin, [(0, 1, 2e-4)])

D0 = spin.DipoleVector[0]  # meV

distance_nm = np.linalg.norm(
    (spin.AtomPositions[1] - spin.AtomPositions[0])
    * spin.latticeLengthXY
)

print(f"Spin separation: {distance_nm:.3f} nm")
print(f"D0:              {D0:.6e} meV")
print(f"D0/h:            {D0 * spin.meVtoGHzConversion * 1e3:.3f} MHz")

## Zero-field spectrum



In [ ]:
hamil.build_Hamiltonian(spin)
hamil.calc_EigEnergies(spin)

return_dict = hamil.plot_EigenMatrix(spin, plot_type="p")
return_dict = hamil.plot_EnergyVsS(spin, type= "z", energyscale= "GHz")

## Magnetic-field dependence

Sweep the external field along z, perpendicular to the spin separation.
The diagram shows the competition between Zeeman and dipolar interactions.

Within an exactly degenerate subspace, the numerical eigenvectors are
not unique.

In [ ]:
B_array = np.linspace(0.0, 0.02, 201)  # T

zeeman_result = hamil.plot_ZeemanDiagramm(
    spin,
    B_array=B_array,
    energyscale="GHz",
)